### Load configurable catalog and reference schema names for the project.

In [0]:
# Load the catalog name from Spark config, or use the default project catalog.
catalog_name = spark.conf.get("training_0002_ecommerce.catalog_name", "training_0002_ecommerce")
reference_schema = spark.conf.get("training_0002_ecommerce.reference_schema", "reference")

# Create the catalog if it does not already exist, then set it as the active catalog.
# Run the Spark SQL statement needed for this setup or transformation step.
spark.sql(f"CREATE CATALOG IF NOT EXISTS {catalog_name}")
spark.sql(f"USE CATALOG {catalog_name}")

### Create the medallion schemas plus the reference schema used by lookup tables.

In [0]:
# Run the Spark SQL statement needed for this setup or transformation step.
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {catalog_name}.bronze")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {catalog_name}.silver")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {catalog_name}.gold")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {catalog_name}.{reference_schema}")

# Show a sample of the result so it can be visually checked.
display(spark.sql(f"SHOW DATABASES FROM {catalog_name}"))

### Create the source-data schema and volume used by Bronze ingestion notebooks.

In [0]:
# Import the dependency used by the lines below.
from pathlib import Path, PurePosixPath

# Load the source schema name from Spark config, or use the default source schema.
source_schema = spark.conf.get("training_0002_ecommerce.source_schema", "source_data")
source_volume = spark.conf.get("training_0002_ecommerce.source_volume", "raw_data")
# Build the full Unity Catalog volume path used to store raw files.
volume_path = f"/Volumes/{catalog_name}/{source_schema}/{source_volume}"

# Create the schema and managed volume, then prepare the nested order_items folder.
# Run the Spark SQL statement needed for this setup or transformation step.
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {catalog_name}.{source_schema}")
spark.sql(f"CREATE VOLUME IF NOT EXISTS {catalog_name}.{source_schema}.{source_volume}")

# Create the target folder in the Unity Catalog volume if it is missing.
dbutils.fs.mkdirs(f"{volume_path}/order_items")

# Print a small status message so the notebook run is easier to follow.
print(f"Source volume ready at {volume_path}")

### Upload local CSV files from the repo into the Unity Catalog volume.

In [0]:
# Get the current notebook path so the repo root can be derived.
notebook_path = dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get()
repo_root = Path((PurePosixPath("/Workspace") / PurePosixPath(notebook_path)).parent.parent)
local_data_dir = repo_root / "0_data"
local_order_items_dir = local_data_dir / "order_items"

# Stop early if the notebook is not being run from the expected Databricks Repo path.
# Check whether the validation condition is met before continuing.
if not local_data_dir.exists():
    # Stop execution with a clear error message when the validation fails.
    raise FileNotFoundError(
        f"Could not find {local_data_dir}. Run this notebook from the Databricks Repo that contains the project files."
    )

# Initialize a list to track which files were uploaded during setup.
copied_files = []

# Copy top-level dimension CSV files into the root of the raw-data volume.
# Iterate through the available records for this step.
for csv_file in sorted(local_data_dir.glob("*.csv")):
    # Copy the local file into the Unity Catalog volume used by Bronze ingestion.
    dbutils.fs.cp(f"file:{csv_file.as_posix()}", f"{volume_path}/{csv_file.name}", True)
    # Record the uploaded file name for the setup summary.
    copied_files.append(csv_file.name)

# Copy daily order-item CSV files into the order_items subfolder.
# Iterate through the available records for this step.
for csv_file in sorted(local_order_items_dir.glob("*.csv")):
    # Copy the local file into the Unity Catalog volume used by Bronze ingestion.
    dbutils.fs.cp(f"file:{csv_file.as_posix()}", f"{volume_path}/order_items/{csv_file.name}", True)
    # Record the uploaded file name for the setup summary.
    copied_files.append(f"order_items/{csv_file.name}")

# Print a small status message so the notebook run is easier to follow.
print(f"Copied {len(copied_files)} CSV files into {volume_path}")
# Show a sample of the result so it can be visually checked.
display(spark.createDataFrame([(name,) for name in copied_files], ["uploaded_file"]))

In [0]:
# spark.sql(f"DROP CATALOG IF EXISTS {catalog_name} CASCADE")